# Genome-wide association study: a full analysis

**Purpose.** Run an association study end to end and, more importantly, learn to distrust
the first result. Most of this exercise is about the things that make a GWAS wrong —
population structure, relatedness, poor QC — and about how you detect them.

**What you will do**
 - **A.** run a first GWAS and set a significance threshold
 - **B.** check it with a QQ plot, which is how inflation shows up
 - **C.** QC the data and see what the association looks like afterwards
 - **D.** run PCA, read the Tracy-Widom statistics, and fit a linear mixed model with
   `regenie` **with and without** the top 20 PCs, to see what correcting for structure does
 - **E.** gene-based testing with SKAT and ACAT for rare variants, where single-variant
   tests have no power
 - **F.** calculate GWAS power for quantitative and binary traits

**The data.** `European_1w` — a European cohort in plink format with a quantitative
phenotype, plus precomputed PCA output, annotation, set-list and mask files for the
gene-based tests. **Called genotypes.**

> ## ⚠️ The data for this exercise is not on this server
>
> The notebook was written for a course machine where each student had the files under
> `/home/student/USER/GWAS/data/`. That folder does not exist here, and the dataset is not
> in any of the course archives — the `chinacourse2025` zip for this day contains only the
> notebook and the lecture PDFs.
>
> The exercise is included **unchanged** so the material is not lost. All the paths are
> collected in the setup cell below, so when the data is found, pointing `DATA` at it is the
> only edit needed.
>
> Everything else about the exercise is intact: the commands, the questions and the
> explanations are exactly as taught.

## Setup

All the paths used by this exercise are set in the cell below.

In [ ]:
#############################################################
# ALL PATHS ARE SET HERE
# If the data moves, this is the ONLY cell you need to change.
# No cell below this one uses a full path.
#############################################################

# Where the GWAS data lives.
#
# NOTE: this dataset is NOT currently on this server. It was written for a
# course machine where every student had a personal copy under
#     /home/student/<username>/GWAS/data/
# Point DATA at the files once they are available and everything below will run.
DATA=$HOME/GWAS/data

# The conda environment the course machine used. Also not present here.
CONDA_ACTIVATE=$HOME/miniconda3/bin/activate
CONDA_ENV=gwas

# where you will do the exercise
WORK_DIR=$HOME/gwas_analysis_human
mkdir -p $WORK_DIR
echo $WORK_DIR > $HOME/.gwas_analysis_workdir
cd $WORK_DIR

# R and Python cannot read bash variables, so write the paths to a file they can read
cat > $WORK_DIR/env.sh <<EOF
export DATA=$DATA
export WORK=$WORK_DIR
EOF

if [ -d "$DATA" ]; then
    echo "data folder: $DATA"
    ls $DATA | head
else
    echo "NOTE: the data folder $DATA was not found."
    echo "This exercise needs the European_1w dataset, which is not on this server yet."
fi

In [ ]:
# R cannot source env.sh, so read the paths out of it rather than repeating them
env <- readLines(path.expand("~/gwas_analysis_human/env.sh"))
getvar <- function(k) sub(paste0('^export ', k, '='), '', grep(paste0('^export ', k, '='), env, value = TRUE)[1])
DATA <- getvar("DATA"); WORK <- getvar("WORK")
setwd(WORK)

In [ ]:
# Python reads the same file, so the paths are still only set in the cell above
import os
env = dict(l.strip().removeprefix("export ").split("=", 1)
           for l in open(os.path.expanduser("~/gwas_analysis_human/env.sh")) if "=" in l)
DATA, WORK = env["DATA"], env["WORK"]
os.chdir(WORK)

# Guide to the notebook

## Interface

- Click a cell and press **Ctrl-Enter** or **Shift-Enter** will exectute the commands in cell
- If you get an unexpected result try to rerun the cell

# Practical — Genome-Wide Association Study (GWAS)

In this practical you will learn the essentials of running a Genome-Wide Association Study (GWAS), along with common pitfalls to avoid.

We will work **entirely from the command line** (Calysto Bash terminals) for data processing with **PLINK/REGENIE/LOCUSZOOM**, and switch to **R/Python** for visualisation.

* **PLINK documentation**  
  * [Original release](http://zzz.bwh.harvard.edu/plink/)  
  * [Current 1.9/2.0 versions](https://www.cog-genomics.org/plink/1.9/)

---

## How this notebook is organised

Each cell is either:

* **Markdown text** – like the block you are reading now.  
* **Calysto Bash** cell – runs Linux shell commands (labelled “Calysto Bash” at top-right).  
* **R** cell – runs R code (labelled “R4.4”).
* **Python** cell – runs Python code (labelled “Python 3.12”).

### Running cells

| Action | Windows / Linux | macOS |
|--------|-----------------|-------|
| Run selected cell | **Shift + Enter** | **Cmd + Enter** |
| Re-run after editing | **Ctrl + Enter** | **Cmd + Enter** |

> **Tip:** click inside a code cell before executing it.

---

Let’s start by inspecting your home directory.  
In the **Calysto Bash** cell below, list its contents:


In [ ]:
# List files in your home folder
ls ~/

In [ ]:
# create the output folder for storing output files
if [ ! -d ./output ]; then
  mkdir ./output
fi

List the content of the data folder that you will used next:

In [ ]:
ls $DATA/

# Exercise A: running your first GWAS

**We're using simnulated UKBiobank data!!!**

Briefly, the GWAS data consist of SNP genotyping data from 10,000 European individuals (someone tells you that they are Europeans but you have to check by yourself) with information about their standing height (in meters).

To make sure the GWAS analyses will run fast the main data file (European_1w.bed) is in a binary format, which is not very reader friendly. 

However, PLINK (the program we will use to run the analyses) will print summary statistics about the data (number of SNPs, number of individuals etc) to the screen when you run an analysis.

Also, there are two additional data files, European_1w.bim and European_1w.fam, which are not in binary format and which contains information about the SNPs in the data and the individuals in the data, respectively.

(You can read more about the data format in the manuals linked to above - but for now this is all you need to know).

Let's look inside the .fam file, which contains information about the individuals. The head and tail commands shows the first and last 10 lines of the file. Try to run them:

### Note:

These 1w European samples were randomly sampled from the whole UKB cohort using stratified random sampling, in which 30% were sampled from the European population and the rest 70% sampled from the non-European population.

In [ ]:
#head prints the 10 lines of the fam file
head $DATA/European_1w.fam
echo "-------------"
#tail prints the last 10 lines of the fam file
tail $DATA/European_1w.fam

In [ ]:
data <-read.table(file.path(DATA, "European_1w.fam"))
str(data)
hist(as.numeric(data$V6),main="Standing Height")

- Standing height is the phenotype here. Does the histogram look roughly normal, and why does that matter for a linear regression of height on genotype?

In [ ]:
#head prints the 10 lines of the bim file
head $DATA/European_1w.bim|column -t
echo "-------------"
#tail prints the last 10 lines of the bim file
tail $DATA/European_1w.bim|column -t

- The `.bim` file has one line per variant. Which of its columns would you need to check before combining this dataset with another one?

In [ ]:
# Because we can not directly view a binary file, we convert the European_1w.bed file to the European_1w.ped to learn the content within it 
# Command for the convertion: plink --bfile $DATA/European_1w --recode tab --memory 5000 --out $DATA/test
# print the 20 rows of the ped file which is reader-friendly
echo 'sample-info columns (1-6) and three SNP (7-8, 9-10, 11-12)'
head -n 20 $DATA/European_1w.ped |
  awk '{print $1,$2,$3,$4,$5,$6,$7,$8,$9,$10,$11,$12}'

Let's try to perform a GWAS of our data, i.e. test each SNP for association with the standing height.

And let's try to do it using linear regression for standing height data

The PLINK option "--bfile $DATA/European_1w" will specify that the data PLINK should analyse are the files in folder called "$DATA/" with the prefix "European_1w".

"—linear" specifies that we want to perform GWAS using linear regression

"—adjust" tells PLINK to output a file that includes p-values that are adjusted for multiple testing using Bonferroni correction as well as other fancier methods.

"—autosome" only use autosomes and not X,Y,MT

Now perform the logistic regression on all the SNPs int the dataset using these options in PLINK by typing:

In [ ]:
# Start running a GWAS using PLINK2  
# running about 3 minutes
source $CONDA_ACTIVATE $CONDA_ENV
plink \
  --bfile $DATA/European_1w \
  --linear \
  --adjust \
  --autosome \
  --memory 5000 \
  --out ./output/European_1w_standing_height
conda deactivate

Take a look at the text PLINK prints to your screen. Specifically, note the

 - number of SNPs

 - number of individuals

 - number of phenotype values
 


In [ ]:
echo "first eight lines"
head ./output/European_1w_standing_height.assoc.linear
echo "significant variants"
awk '$NF<5e-8' ./output/European_1w_standing_height.assoc.linear|head

Next, plot the results of the GWAS using the following command run in R



In [ ]:
# running about 1 minutes
options(repr.plot.width = 16,
        repr.plot.height = 4,
        repr.plot.res = 600)

source(file.path(DATA, "plotPlink.R"))

# plot the results (.assoc.linear is the output file from plink)
plots= plot_qqman(
  plink_assoc_file= "./output/European_1w_standing_height.assoc.linear",
  pheno_name= "Standing_height",
  save_plot = FALSE,
  lambda1_qq_pos = c(1.48, -5.5),
  lambda2_qq_pos = c(1.1, -4)
)
print(plots$manhattan_plot)

## Identify Significance Thresholds

- Genome-wide significance line – e.g. 5 × 10⁻⁸ for single-variant GWAS or a Bonferroni-corrected line for gene-based tests. Points above it are **significant**.

- Suggestive line (optional) – marks loci worth follow-up but not yet definitive.

A bonferroni corrected p-value threshold based on an initial p-value threshold of 0.05 is not shown on the plot. Explain how this threshold was reached and calculate the exact threshold using your knowledge of how many SNPs you have in your dataset (NB if you want to calculate log10 in R you can use the function log10).

In [ ]:
wc -l $DATA/European_1w.bim

In [ ]:
from math import log10
print(0.05 / 784_256)
-log10(0.05 / 784_256)

- Using this threshold, does any of the SNPs in your dataset seem to be associated with standing height?

- Do your results seem plausible? Why/why not?

In [ ]:
awk 'NR>1 && $5 < 6.375469234535663e-08' ./output/European_1w_standing_height.assoc.linear.adjusted | wc -l
awk 'NR>1 && $5 < 5e-08' ./output/European_1w_standing_height.assoc.linear.adjusted | wc -l


# Exercise B: checking if it went OK using QQ-plot

Now look at the QQ-plot that you already generated (second plot above). Here the red line is the x=y line and the thin curves are a confidence band.

- What does this plot suggest and why?


In [ ]:
# running about 0.5 minutes
options(repr.plot.width = 4.1,
        repr.plot.height = 4.1,
        repr.plot.res = 600)
print(plots$qq_plot)

- Axes

    - X-axis: Expected −log₁₀ P under the null.

    - Y-axis: Observed −log₁₀ P from your test.

- Diagonal (= null)

    Points near the diagonal imply well-calibrated statistics.

- Genomic inflation factor (λ<sub>GC</sub>)

    λ ~ 1 → good; λ ≫ 1 → inflation; λ < 1 → conservative.

Use the QQ-plot together with the Manhattan plot: Manhattan tells where the signals are, QQ-plot tells whether your test statistics are globally trustworthy.

# Exercise C: QC your data

As you can see, a lot can go wrong if you do not check the quality of your data before running your GWAS! So if you want meaningful/useful output you always have to run a lot of quality checks (QC) before running the association tests. We will try to go through some useful QC steps now.

One potential problem in association studies is spurious relatedness, where some of the individuals in the sample are closely related. Closely related individuals can be inferred with PLINK using the following command, which only uses autosomal SNPs with a minor allele frequency > 5%:



In [ ]:
# running about 2 minutes
source $CONDA_ACTIVATE $CONDA_ENV
plink \
  --bfile $DATA/European_1w \
  --genome \
  --autosome \
  --maf 0.05 \
  --memory 5000 \
  --out ./output/European_1w_genome
conda deactivate

In [ ]:
head ./output/European_1w_genome.genome

- `--genome` gives one line per pair of individuals. Roughly how many lines is that for 10,000 individuals, and why does this step take so much longer than the association test?

The results can be summarised in a plot with the following Python code and gives the names of potential related pairs: 

In [ ]:
# running about 1.5 minutes
import sys
sys.path.append(DATA)
from plotPlink import plot_ibd

plot_ibd("./output/European_1w_genome.genome")

- Where in this plot would a parent-offspring pair sit, and where would a pair of unrelated individuals sit?
- If you found related pairs, would you drop one of each pair or model the relatedness instead? What do you lose either way?

The figure shows estimates of the relatedness for all pairs of individuals.

For each pair k1 is the proportion of the genome where the pair shares 1 of their alleles identical-by-descent (IBD) and k2 is the proportion of the genome where the pair shares both their alleles IBD.

- The expected (k1,k2) values for simple relationships are shown in the figure. Are any of the individuals in your dataset closely related?

- What assumption in association studies is violated when individuals are related? 

- And last but not least: how would you recognize if the same person is included twice? (this actually happens often!)


We usually only remove 1. or 2. degree relatives (MZ,PO,FS,HS) from the analysis or we use a mixed model to take the relatedness into account. 

**Principal component analysis (PCA)** and a very similar methods called **multidimensional scaling** is also often used to reveal problems in the data.

Such analyses can be used to project all the genotype information (e.g. 500,000 marker sites) down to a low number of dimensions e.g. two.

## Multidimensional scaling

Multidimensional scaling based on your data can be performed with PLINK as follows (the option --mind is used to remove the few individuals which have more than 20% missingness):

In [ ]:
# running about 3 minutes
source $CONDA_ACTIVATE $CONDA_ENV
plink \
  --bfile $DATA/European_1w \
  --cluster \
  --mds-plot 2 \
  --mind 0.05 \
  --memory 5000 \
  --out ./output/European_1w_mds
conda deactivate

- MDS and PCA are being run on the same genotypes. Both are used here to look for structure — what would you expect the first dimension to separate the individuals by?

Try to plot the results in Python:

In [ ]:
import os
import sys
sys.path.append(DATA)
from plotPlink import plot_clusters

# plot the results (plink.mds is the output file from plink)
plot_clusters(
    "./output/European_1w_mds.mds",
    os.path.join(DATA, "European_1w.fam")
)

It shows the first two dimensions and each individual is represented by a point.

Clustering of individuals with similar trait values may indicate batch bias or population structure.
- Do you see any clustering or gradients related to the trait values?
- What else could explain such patterns?

Let's perform PCA.

## PCA

Principal components analysis (PCA) is one of the most useful techniques to visualise genetic diversity in a dataset. The methodology is not restricted to genetic data, but in general allows breaking down high-dimensional datasets to two or more dimensions for visualisation in a two-dimensional space.

### Preparing the parameter file

For actually running the analysis, we use a software called **smartPCA** from the Eigensoft package. As many other tools from this and related packages, smartPCA reads in a parameter file which specifies its input and output files and options. The basic format of the parameter file looks like this:

In [ ]:
cat $DATA/European_1w_smartpca.par

Here, the first three parameters specify the input genotype files, as discussed above.

The next two rows specify two output file names, typically with ending *.evec and *.eval.

numoutevec specifies the number of principal components that we compute.


In [ ]:
# # do not run!
# # running about 95 minutes
# source $CONDA_ACTIVATE $CONDA_ENV
# start_time=$(date +%s)
# smartpca -p \
#   $DATA/European_1w_smartpca.par \
#   > ./output/European_1w_pca.log
# end_time=$(date +%s)
# elapsed_seconds=$((end_time - start_time))
# echo "Elapsed time: $(($elapsed_seconds / 60)) mins"
# conda deactivate

## Let's examine the pre-run results

In [ ]:
# check the log
awk 'NR > 16 && NR < 35' $DATA/European_1w_pca.txt

- The Tracy-Widom test says 676 PCs are significant, but only the top 20 are used as covariates. What is the cost of using too few, and of using too many?

In [ ]:
# top 10 rows of .eigenvec
head -n 10 $DATA/European_1w.eigenvec | column -t
# the first line is the eigenvalues for the requested 20 eigenvec

In [ ]:
# top 20 rows of .eigenval
head -n 20 $DATA/European_1w.eigenval

Let's visualize the results of PCA.

In [ ]:
import os
# running about 0.5 minutes
import sys
sys.path.append(DATA)
from plotPCA import plot_pca_plots

plot_pca_plots(
    eigenvec_file=os.path.join(DATA, "European_1w_fig.eigenvec"),
    country_file=os.path.join(DATA, "European_1w_phenotypes.txt"),
    save_figs=False
)

It shows the first several PCs capture **very strong population structure**.

The dataset exhibits pronounced population structure.

These PCs can be safely used as covariates in downstream GWAS.

## Tracy-Widom statistics

The twstats program computes Tracy-Widom statistics to evaluate the statistical significance of each principal component identified by pca.  

In [ ]:
# the head 8 rows of Tracy-Widom statistics
awk 'NR > 35 && NR < 46' $DATA/European_1w_pca.txt

- A significant Tracy-Widom statistic means a PC captures more structure than chance. Why does that not automatically mean the PC is worth correcting for?

In [ ]:
awk 'NR >= 38 && NR <= 10036 && $5 != "NA" && ($5+0) < 0.05' $DATA/European_1w_pca.txt | wc -l

Among the top 676 principal components (PCs) that showed statistical significance, we retained only the top 20 PCs for downstream analyses as an illustrative example."

In [ ]:
head $DATA/covar_PCs.txt | column -t

Adiitionally, let's try to fix the issue by filtering SNPs. We can remove many of the error prone SNPs by removing

- SNPs that are not in HWE (Hardy weinberg Equilibrium) (option --hwe)

- the rare SNPs (difficult to genotype and very error prone) (option --maf)

- SNPs with lots of missing data (why?) (option --geno)

**Try to redo the above plink analysis by adding the additional filters**

--hwe 0.0001 --maf 0.05 --geno 0.05

which remove sites not in HWE (p-value 0.0001), low minor allele frequency (<5%), high genotype missingness (>5%).

- Can you now see patterns or gradients among individuals with respect to the quantitative trait?

Let us try to rerun an association analysis with these additional filters (and a new output name so we won't overwrite our old results). 

In [ ]:
# # do not run!
# # running about 4 hours
# source $CONDA_ACTIVATE $CONDA_ENV
# plink \
#   --bfile $DATA/European_1w \
#   --linear hide-covar \
#   --autosome \
#   --memory 5000 \
#   --out ./output/European_1w_QC_PC20 \
#   --hwe 0.0001 \
#   --maf 0.05 \
#   --geno 0.05 \
#   --covar $DATA/covar_PCs.txt \
#   --covar-name PC1-PC20
# conda deactivate

Now try to plot the manhattan plot and the qqplot in R:

In [ ]:
# running about 1 minutes
options(repr.plot.width = 16,
        repr.plot.height = 4,
        repr.plot.res = 600)
source(file.path(DATA, "plotPlink.R"))

# plot the results (.assoc.linear is the output file from plink)
plots= plot_qqman(
  plink_assoc_file= file.path(DATA, "European_1w_QC_PC20.assoc.linear"),
  pheno_name= "Standing_height",
  save_plot = FALSE,
  lambda1_qq_pos = c(1.48, -5.5),
  lambda2_qq_pos = c(1.1, -4)
)
print(plots$manhattan_plot)

In [ ]:
# running about 0.5 minutes
options(repr.plot.width = 4.1,
        repr.plot.height = 4.1,
        repr.plot.res = 600)
print(plots$qq_plot)

- How does the QQ plot look now - any signs of inflation?
- How many genome wide significant SNPs?

Information about the most significant SNPs is printed below.

Identify the chromosome, physical position (BP), beta and SNP name. 

In [ ]:
head $DATA/European_1w_QC_PC20.assoc.linear
awk 'NR>1 {print $1,$2,$3,$9}' \
  $DATA/European_1w_QC_PC20.assoc.linear | sort -k4,4g 2>/dev/null | head -1 || true

Let's try to plot the region with the most significant SNP in a 1 Mb window around this SNP:

In [ ]:
awk '{$1=$1}1' OFS='\t' $DATA/European_1w_QC_PC20.assoc.linear > ./output/European_1w_QC_PC20.assoc_linear.txt
source $CONDA_ACTIVATE $CONDA_ENV
$DATA/../locuszoom/bin/locuszoom \
    --metal ./output/European_1w_QC_PC20.assoc_linear.txt \
    --markercol SNP \
    --pvalcol P \
    --refsnp rs7808919 \
    --chr 7 \
    --flank 500kb \
    --pop EUR \
    --build hg19 \
    --source 1000G_March2012 \
    --prefix ./output/European_1w
conda deactivate

Open the figure manually using this path:

**./output/European_1w_250716_rs7808919/chr7_7402687-8402687.pdf**

# Exercise D: linear mixed model
Below you will apply a method called [regenie](https://rgcgithub.github.io/regenie/) that implemented linear mixed model for GWAS

## Step0: Format the data as needed


In [ ]:
head $DATA/phenotype.txt

#  Part 1: perform regenie analyses *without* considering the top 20 pcs

## Step1: fitting the null linear mixed model with regenie

For quantitative traits (such as standing height), regenie fits a linear mixed model by default.  
It is recommended to inverse normalize the phenotype before analysis to improve normality and statistical power.

In [ ]:
# running about 2 minutes
source $CONDA_ACTIVATE $CONDA_ENV
regenie \
  --step 1 \
  --bed $DATA/European_1w_QC \
  --phenoFile $DATA/phenotype.txt \
  --strict \
  --bsize 1000 \
  --loocv \
  --lowmem \
  --lowmem-prefix ./output/regenie_tmp_preds \
  --apply-rint \
  --out ./output/regenie_step1_WO_PC20
conda deactivate

See more parameter explanations: https://rgcgithub.github.io/regenie/options/.

## Step 2: performing single-variant association tests

For quantitative traits (such as standing height), regenie automatically uses a linear mixed model for association testing.

There is no need for saddle point approximation or Firth correction, as these are specific to binary traits.

The output will include effect sizes, standard errors, and p-values for each variant.

In [ ]:
# running about 0.5 minutes
source $CONDA_ACTIVATE $CONDA_ENV
regenie \
  --step 2 \
  --bed $DATA/European_1w_QC \
  --ref-first \
  --phenoFile $DATA/phenotype.txt \
  --strict \
  --bsize 1000 \
  --apply-rint \
  --pred ./output/regenie_step1_WO_PC20_pred.list \
  --out ./output/regenie_step2_asso_WO_PC20
conda deactivate

- These are the results *without* the principal components. Look at the QQ plot below — how far off the diagonal is it?

### View the result of REGENIE-GWAS


In [ ]:
head ./output/regenie_step2_asso_WO_PC20_phenotype.regenie | column -t

In [ ]:
# search for the most significant SNP
awk 'NR>1 {print $1,$2,$3,$12}' \
  ./output/regenie_step2_asso_WO_PC20_phenotype.regenie | sort -k4,4gr 2>/dev/null | head -1 || true

In [ ]:
# Prepare for plotting the results
regenie_results= data.table::fread("./output/regenie_step2_asso_WO_PC20_phenotype.regenie")
regenie_results[, P := 10^(-LOG10P)] %>%
    data.table::setnames(
        c("CHROM", "GENPOS", "ID"),
        c("CHR", "BP", "SNP")
    )
regenie_results %>%
    data.table::fwrite("./output/regenie_step2_asso_WO_PC20_phenotype_v2.regenie")

In [ ]:
# running about 0.5 minutes
options(repr.plot.width = 16,
        repr.plot.height = 4,
        repr.plot.res = 600)
source(file.path(DATA, "plotPlink.R"))

# plot the results (.regenie is the output file from regenie)
plots= plot_qqman(
  plink_assoc_file= "./output/regenie_step2_asso_WO_PC20_phenotype_v2.regenie",
  pheno_name= "Standing_height",
  save_plot = FALSE,
  lambda1_qq_pos = c(1.48, -5.5),
  lambda2_qq_pos = c(1.1, -4)
)
print(plots$manhattan_plot)

In [ ]:
# running about 0.5 minutes
options(repr.plot.width = 4.1,
        repr.plot.height = 4.1,
        repr.plot.res = 600)
print(plots$qq_plot)

#  Part 2: perform regenie analyses *with* considering the top 20 pcs

## Step1: fitting the null linear mixed model with regenie

For quantitative traits (such as standing height), regenie fits a linear mixed model by default.  
It is recommended to inverse normalize the phenotype before analysis to improve normality and statistical power.

In [ ]:
# running about 2 minutes
source $CONDA_ACTIVATE $CONDA_ENV
regenie \
  --step 1 \
  --bed $DATA/European_1w_QC \
  --phenoFile $DATA/phenotype.txt \
  --covarFile $DATA/covar_PCs.txt \
  --strict \
  --bsize 1000 \
  --loocv \
  --lowmem \
  --lowmem-prefix ./output/regenie_tmp_preds \
  --apply-rint \
  --out ./output/regenie_step1_W_PC20
conda deactivate

See more parameter explanations: https://rgcgithub.github.io/regenie/options/.

## Step 2: performing single-variant association tests

For quantitative traits (such as standing height), regenie automatically uses a linear mixed model for association testing.

There is no need for saddle point approximation or Firth correction, as these are specific to binary traits.

The output will include effect sizes, standard errors, and p-values for each variant.

In [ ]:
# running about 0.5 minutes
source $CONDA_ACTIVATE $CONDA_ENV
regenie \
  --step 2 \
  --bed $DATA/European_1w_QC \
  --covarFile $DATA/covar_PCs.txt \
  --ref-first \
  --phenoFile $DATA/phenotype.txt \
  --strict \
  --bsize 1000 \
  --apply-rint \
  --pred ./output/regenie_step1_W_PC20_pred.list \
  --out ./output/regenie_step2_asso_W_PC20
conda deactivate

- Compare this Manhattan and QQ plot with the run without PCs. Which peaks survived, and which disappeared?
- A peak that vanishes once you correct for ancestry: was it never real, or could correcting have removed a true signal? How would you tell?

In [ ]:
# running about 0.5 minutes
source $CONDA_ACTIVATE $CONDA_ENV
regenie \
  --step 2 \
  --bed $DATA/v2_backup_20250718/European_1w_imp \
  --covarFile $DATA/covar_PCs.txt \
  --chr 22\
  --ref-first \
  --phenoFile $DATA/phenotype.txt \
  --strict \
  --bsize 1000 \
  --apply-rint \
  --pred ./output/regenie_step1_W_PC20_pred.list \
  --out ./output/regenie_step2_asso_W_PC20
conda deactivate

### View the result of REGENIE-GWAS


In [ ]:
head ./output/regenie_step2_asso_W_PC20_phenotype.regenie | column -t

In [ ]:
# search for the most significant SNP
awk 'NR>1 {print $1,$2,$3,$12}' \
  ./output/regenie_step2_asso_W_PC20_phenotype.regenie | sort -k4,4gr 2>/dev/null | head -1 || true

In [ ]:
# Prepare for plotting the results
library(magrittr)
regenie_results= data.table::fread("./output/regenie_step2_asso_W_PC20_phenotype.regenie")
regenie_results[, P := 10^(-LOG10P)] %>%
    data.table::setnames(
        c("CHROM", "GENPOS", "ID"),
        c("CHR", "BP", "SNP")
    )
regenie_results %>%
    data.table::fwrite("./output/regenie_step2_asso_W_PC20_phenotype_v2.regenie")

In [ ]:
# running about 0.5 minutes
options(repr.plot.width = 16,
        repr.plot.height = 4,
        repr.plot.res = 600)
source(file.path(DATA, "plotPlink.R"))

# plot the results (.regenie is the output file from regenie)
plots= plot_qqman(
  plink_assoc_file= "./output/regenie_step2_asso_W_PC20_phenotype_v2.regenie",
  pheno_name= "Standing_height",
  save_plot = FALSE,
  lambda1_qq_pos = c(1.48, -5.5),
  lambda2_qq_pos = c(1.1, -4)
)
print(plots$manhattan_plot)

In [ ]:
# running about 0.5 minutes
options(repr.plot.width = 4.1,
        repr.plot.height = 4.1,
        repr.plot.res = 600)
print(plots$qq_plot)

# Exercise E : Gene-based testing

Instead of performing single-variant association tests, multiple variants can be aggregated in a given region, such as a gene.

This can be especially helpful when testing **rare variants** as single-vatiant tests usuaally have lower power performance.

To avoid inflation in the gene-based tets due to rare variants as well as reduce computation time, we can implement the collapsing approach of gene-based testing proposed in SAIGE-GENE+, where ultra-rare variants are aggregated into a mask.

### Annotation input files: to define variant sets and functional annotations which will be used to generate masks.

Each line contains the variant name, the set/gene name and a single annotation category (space/tab separated).

Variants not in this file will be assigned to a default "NULL" category. A maximum of 63 annotation categories (+NULL category) is allowed.

To obtain a single annotation per gene, we could choose the most deleterious functional annotation across the gene transcripts or alternatively use the canonical transcript (note that its definition can vary across software).

In [ ]:
head $DATA/anno_file.txt | column -t

### Set list file: to list variants within each set/gene to use when building masks.

Each line contains the set/gene name followed by a chromosome and physical position for the set/gene, then by a comma-separated list of variants included in the set/gene.


In [ ]:
head $DATA/set_list.txt | column -t

### Mask file

This file specifies which annotation categories should be combined into masks.

Each line contains a mask name followed by a comma-separated list of categories included in the mask (i.e. union is taken over categories).

In [ ]:
head $DATA/mask_file.txt | column -t

### Checking input files

To assess the concordance between the input files for building masks, we can use **--check-burden-files** which will generate a report in **file_masks_report.txt** containing:

- for each set, the list the variants in the set-list file which are unrecognized (not genotyped or not present in annotation file for the set)

- for each mask, the list of annotations in the mask definition file which are not in the annotation file

Additionally, we can use **--strict-check-burden** to enforce full agreement between the three files (if not, program will terminate) :

- all genotyped variants in the set list file must be in the annotation file (for the corresponding set)

- all annotations in the mask definition file must be present in the annotation file

In [ ]:
source $CONDA_ACTIVATE $CONDA_ENV
regenie \
  --step 2 \
  --bed $DATA/European_1w \
  --covarFile $DATA/covar_PCs.txt \
  --ref-first \
  --phenoFile $DATA/phenotype.txt \
  --strict \
  --bsize 1000 \
  --apply-rint \
  --pred ./output/regenie_step1_W_PC20_pred.list \
  --check-burden-files \
  --anno-file $DATA/anno_file.txt \
  --set-list $DATA/set_list.txt \
  --mask-def $DATA/mask_file.txt \
  --skip-test \
  --strict-check-burden \
  --out ./output/burden_check
conda deactivate

In [ ]:
source $CONDA_ACTIVATE $CONDA_ENV
regenie \
  --step 2 \
  --bed $DATA/v2_backup_20250718/European_1w_imp \
  --covarFile $DATA/covar_PCs.txt \
  --chr 22\
  --ref-first \
  --phenoFile $DATA/phenotype.txt \
  --strict \
  --bsize 1000 \
  --apply-rint \
  --pred ./output/regenie_step1_W_PC20_pred.list \
  --check-burden-files \
  --anno-file $DATA/anno_file.txt \
  --set-list $DATA/set_list.txt \
  --mask-def $DATA/mask_file.txt \
  --skip-test \
  --strict-check-burden \
  --out ./output/burden_check
conda deactivate

### AAF file

Both functional annotations and alternative allele frequency (AAF) cutoffs are used when building masks (e.g. only considering LoF sites where AAF is below 1%).

By default, the AAF for each variant is computed from the sample but alternatively, the user can specify variant AAFs using this file.

### AAF cutoffs

Option **--aaf-bins** specifies the AAF upper bounds used to generate burden masks (AAF and not MAF [minor allele frequency] is used when deciding which variants go into a mask).

By default, a mask based on singleton sites are always included.

For example, **--aaf-bins 0.01,0.05** will generate 3 burden masks for AAFs in [0,0.01], [0,0.05] and singletons.

## SKAT/ACAT tests

The option **--vc-tests** is used to specify the gene-based tests to run. By default, these tests use all variants in each mask category.

If you'd like to only include variants whose AAF is below a given threshold ,e.g. only including rare variants, you can use --vc-maxAAF.

For example, **--vc-tests skato,acato-full** will run SKATO and ACATO (both using the default grid of 8 rho values for the SKATO models) and the p-values for SKAT, SKATO, ACATV and ACATO will be output.

**Ultra-rare variants** (defined by default as MAC ≤ 10, see --vc-MACthr) are collapsed into a burden mask which is then included in the tests instead of the individual variants.

## Joint test for burden masks

The ACAT test combines the p-values of the individual burden masks using the Cauchy combination method.

If you only want to output the results for the joint tests (ignore the marginal tests), use **--joint-only**.

In [ ]:
source $CONDA_ACTIVATE $CONDA_ENV
regenie \
  --step 2 \
  --bed $DATA/v2_backup_20250718/European_1w_imp \
  --chr 22 \
  --covarFile $DATA/covar_PCs.txt \
  --ref-first \
  --phenoFile $DATA/phenotype.txt \
  --strict \
  --bsize 1000 \
  --apply-rint \
  --pred ./output/regenie_step1_W_PC20_pred.list \
  --anno-file $DATA/anno_file.txt \
  --set-list $DATA/set_list.txt \
  --mask-def $DATA/mask_file.txt \
  --rgc-gene-p \
  --vc-tests skato,acato-full \
  --joint acat,sbat \
  --vc-MACthr 10 \
  --out ./output/gene_based_testing
conda deactivate

- Gene-based tests combine many rare variants in a gene into one test. Why does that help for rare variants when a single-variant test does not?
- SKAT and burden tests make different assumptions about the direction of effects. Which would you prefer if a gene contains both protective and risk variants?

For each set, this will produce masks using 3 AAF cutoffs (singletons, 5% and 10% AAF).

The masks are written to PLINK bed file (in **_masks.{bed,bim,fam}**) and tested for association with each trait (summary stats in **_phenotype_name.regenie**).

Additionally, a header line is included (starting with ##) which contains mask definition information.

Masks will have name set_name.mask_name.AAF_cutoff with the chromosome and physical position having been defined in the set list file, and the reference allele being ref, and the alternate allele corresponding to **mask_name.AAF_cutoff**.

When using **--rgc-gene-p**, it will apply the single p-value per gene GENE_P strategy using all masks.

## Excercise F: Calculate GWAS power


Now we generate a statistical power analysis plot for GWAS studies.

Supports binary (case-control) traits over a range of odds ratios and minor allele frequencies, and quantitative traits over a range of effect sizes and minor allele frequencies.


### Part1 : Quantitative traits

In our above example, standing height is the quantitative trait.

In [ ]:
import os
# calculate the standard deviation of the quantitative trait
import pandas as pd
df = pd.read_table(
    os.path.join(DATA, "European_1w_phenotypes.txt"),
    sep = '\\s+',
    header = 0
)
print(df.shape)
df['Standing_height'].std(skipna=True)

In [ ]:
options(repr.plot.width = 16,
        repr.plot.height = 8,
        repr.plot.res = 600)
source(file.path(DATA, "plotPlink.R"))
power_results_qt <- plot_gwas_power(
        trait_type = "qt",
        sd_trait = 0.09365788681305078,
        N = 10000,
        maf_levels = c(0.01, 0.02, 0.05, 0.10, 0.20, 0.50),
        effect_size = seq(0.01, 0.10, 0.001),
        save_plot = FALSE
    )
print(power_results_qt$plot)

### Part2: Binary traits

Now we generate a statistical power analysis plot for a given range of odds ratios and minor allele frequencies in a **case-control** GWAS study. 

Statistical power is crucial for designing a successful GWAS.

It helps you determine the probability of detecting a true association, given a specific sample size, allele frequency, and effect size (Odds Ratio).

### Example Usage

Let's run the example with a sample dataset:

- Cases: 4,324

- Controls: 93,945

- Odds Ratios: Ranging from 1.01 to 2.00

- MAF: 0.01, 0.02, 0.05, 0.10, 0.20, 0.50

In [ ]:
options(repr.plot.width = 16,
        repr.plot.height = 8,
        repr.plot.res = 600)
source(file.path(DATA, "plotPlink.R"))

power_results <- plot_gwas_power(
        trait_type = "bt",
        n_cases = 4324,
        n_controls = 93945,
        maf_levels = c(0.01, 0.02, 0.05, 0.10, 0.20, 0.50),
        or_range = seq(1.01, 2.00, 0.01),
        save_plot = FALSE
    )
print(power_results$plot)

### Run the cell below to take the quiz

In [ ]:
from jupyterquiz import display_quiz

display_quiz("https://raw.githubusercontent.com/popgenDK/courses/main/current_exercises/gwas/quiz/gwas_analysis.json")
